# JAK2 molecule analysis

Load `data/experiment_history.csv`, rank molecules by the selective-JAK2 objective, and draw the top five structures. The objective is **minimized** (lower is better). When at least five complete docking results are available, rank by the actual objective; otherwise use the saved Chemprop-predicted objective.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from rdkit import Chem
from rdkit.Chem import Draw, QED

# Run this notebook from the project directory.
history_path = Path('data/experiment_history.csv')
if not history_path.exists():
    history_path = Path('experiment_history.csv')

history = pd.read_csv(history_path)
if 'predicted_objective' not in history.columns and 'jak2_objective' in history.columns:
    history['predicted_objective'] = history['jak2_objective']
if history.empty or 'smiles' not in history.columns:
    print(f'No molecule records found in {history_path}.')
else:
    history = history.dropna(subset=['smiles']).copy()
    # Convert score/property columns to numbers; invalid or blank values become NaN.
    numeric_columns = [
        'qed', 'predicted_jak2', 'predicted_lck',
        'docking_jak2', 'docking_lck', 'predicted_objective', 'actual_objective'
    ]
    for column in numeric_columns:
        if column in history.columns:
            history[column] = pd.to_numeric(history[column], errors='coerce')

    # Prefer actual docking when there are at least five fully docked molecules.
    docking_cols = ['docking_jak2', 'docking_lck']
    docking_count = history[docking_cols].dropna().shape[0] if set(docking_cols).issubset(history.columns) else 0

    if docking_count >= 5:
        jak2_col, lck_col = docking_cols
        score_source = 'actual DOCKSTRING docking'
    else:
        jak2_col = lck_col = None
        score_source = None

    if jak2_col is not None:
        ranked = history.dropna(subset=[jak2_col, lck_col]).copy()
        ranked[jak2_col] = pd.to_numeric(ranked[jak2_col], errors='coerce')
        ranked[lck_col] = pd.to_numeric(ranked[lck_col], errors='coerce')
        ranked = ranked.dropna(subset=[jak2_col, lck_col])
        # Paper objective: JAK2 - min(LCK + 8.1, 0) + 10 * (1 - QED).
        # If QED is absent from the CSV, calculate it from the saved SMILES.
        if 'qed' not in ranked.columns:
            ranked['qed'] = float('nan')
        ranked['qed'] = pd.to_numeric(ranked['qed'], errors='coerce').astype(float)
        missing_qed = ranked['qed'].isna()
        ranked.loc[missing_qed, 'qed'] = ranked.loc[missing_qed, 'smiles'].map(
            lambda s: float(QED.qed(Chem.MolFromSmiles(s)))
            if Chem.MolFromSmiles(s) is not None else float('nan')
        )
        ranked = ranked.dropna(subset=['qed'])
        ranked['jak2_objective_for_ranking'] = (
            ranked[jak2_col]
            - (ranked[lck_col] + 8.1).clip(upper=0)
            + 10 * (1 - ranked['qed'])
        )
        ranked['jak2_objective_for_ranking'] = pd.to_numeric(
            ranked['jak2_objective_for_ranking'], errors='coerce'
        )
        ranked = ranked.dropna(subset=['jak2_objective_for_ranking'])
        top5 = ranked.nsmallest(5, 'jak2_objective_for_ranking').copy()
        top5.insert(0, 'rank', range(1, len(top5) + 1))
        top5['score_source'] = score_source
        display_columns = [
            c for c in ['rank', 'jak2_objective_for_ranking', 'score_source',
                        jak2_col, lck_col, 'qed', 'smiles']
            if c in top5.columns
        ]
        print(f'Top {len(top5)} by lowest objective using {score_source}:')
        display(top5[display_columns].round({'jak2_objective_for_ranking': 3, 'qed': 3}))

        molecules, legends = [], []
        for _, row in top5.iterrows():
            mol = Chem.MolFromSmiles(row['smiles'])
            if mol is None:
                continue
            molecules.append(mol)
            legends.append(
                f"Rank {int(row['rank'])} | objective {row['jak2_objective_for_ranking']:.2f}\n"
                f"JAK2 {row[jak2_col]:.2f} | LCK {row[lck_col]:.2f} | QED {row['qed']:.2f}"
            )

        if molecules:
            display(Draw.MolsToGridImage(
                molecules, molsPerRow=2, subImgSize=(420, 300), legends=legends
            ))
        else:
            print('No valid SMILES among the ranked records.')
    elif 'predicted_objective' in history.columns and history['predicted_objective'].notna().any():
        # Fallback for older exports that saved only the computed objective.
        top5 = history.dropna(subset=['predicted_objective']).nsmallest(5, 'predicted_objective')
        print('Ranking by saved predicted_objective (lower is better).')
        display(top5[['predicted_objective', 'smiles']])
        mols = [Chem.MolFromSmiles(s) for s in top5['smiles']]
        valid = [(m, r) for m, (_, r) in zip(mols, top5.iterrows()) if m is not None]
        if valid:
            display(Draw.MolsToGridImage(
                [m for m, _ in valid], molsPerRow=2, subImgSize=(420, 300),
                legends=[f"Predicted objective {r['predicted_objective']:.2f}" for _, r in valid]
            ))
    else:
        print('History has molecule records, but not enough JAK2/LCK scores to calculate the objective.')

## Common features among the top docked molecules

This selects up to five molecules with actual JAK2 and LCK docking scores, ranks them by the project objective (lower is better), and summarizes shared descriptors and substructures. Feature frequency is descriptive only; five molecules are too few to establish which features cause stronger docking.

In [ ]:
from rdkit.Chem import Descriptors, Lipinski, rdMolDescriptors

actual_cols = ['docking_jak2', 'docking_lck']
if not set(actual_cols).issubset(history.columns):
    print('The history does not contain both actual docking score columns.')
else:
    docked = history.dropna(subset=actual_cols + ['smiles']).copy()
    if docked.empty:
        print('No molecules have actual JAK2 and LCK docking scores yet.')
    else:
        if 'qed' not in docked.columns:
            docked['qed'] = float('nan')
        docked['qed'] = pd.to_numeric(docked['qed'], errors='coerce').astype(float)
        missing_qed = docked['qed'].isna()
        docked.loc[missing_qed, 'qed'] = docked.loc[missing_qed, 'smiles'].map(
            lambda s: float(QED.qed(Chem.MolFromSmiles(s)))
            if Chem.MolFromSmiles(s) is not None else float('nan')
        )
        docked = docked.dropna(subset=['qed']).copy()
        docked['objective'] = (
            docked['docking_jak2']
            - (docked['docking_lck'] + 8.1).clip(upper=0)
            + 10 * (1 - docked['qed'])
        )
        # Keep one row per canonical molecule, using its latest recorded result.
        docked['canonical_smiles'] = docked['smiles'].map(
            lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s))
            if Chem.MolFromSmiles(s) is not None else None
        )
        docked['objective'] = pd.to_numeric(docked['objective'], errors='coerce')
        docked = docked.dropna(subset=['objective'])
        top_docked = (docked.dropna(subset=['canonical_smiles'])
                      .drop_duplicates('canonical_smiles', keep='last')
                      .nsmallest(5, 'objective').copy())
        print(f'Top {len(top_docked)} actually docked molecules by objective (lower is better):')
        display(top_docked[['objective', 'docking_jak2', 'docking_lck', 'qed', 'smiles']].round(3))

        mols = [Chem.MolFromSmiles(s) for s in top_docked['canonical_smiles']]
        descriptor_rows = []
        for mol in mols:
            descriptor_rows.append({
                'MW': Descriptors.MolWt(mol),
                'LogP': Descriptors.MolLogP(mol),
                'TPSA': rdMolDescriptors.CalcTPSA(mol),
                'HBD': Lipinski.NumHDonors(mol),
                'HBA': Lipinski.NumHAcceptors(mol),
                'Rotatable bonds': Lipinski.NumRotatableBonds(mol),
                'Aromatic rings': rdMolDescriptors.CalcNumAromaticRings(mol),
                'Fraction sp3': rdMolDescriptors.CalcFractionCSP3(mol),
            })
        descriptors = pd.DataFrame(descriptor_rows)
        if not descriptors.empty:
            print('Physicochemical descriptors (mean across selected molecules):')
            display(descriptors.agg(['mean', 'min', 'max']).round(2))

        # Count common medicinal-chemistry motifs in how many molecules they occur.
        from rdkit import DataStructs
        motifs = {
            'Aromatic ring': 'a1aaaaa1',
            'Nitrogen-containing aromatic ring': '[n]1aaaaa1',
            'Amide': '[NX3][CX3](=[OX1])[#6]',
            'Amine (non-amide N)': '[NX3;!$(NC=O)]',
            'Carbonyl': '[CX3]=[OX1]',
            'Sulfonamide': '[SX4](=[OX1])(=[OX1])[NX3]',
            'Halogen (F, Cl, Br, I)': '[F,Cl,Br,I]',
            'Hydroxyl': '[OX2H]',
        }
        motif_rows = []
        for label, smarts in motifs.items():
            query = Chem.MolFromSmarts(smarts)
            count = sum(mol.HasSubstructMatch(query) for mol in mols)
            motif_rows.append({'Feature': label, 'Molecules containing feature': count,
                              'Fraction': count / len(mols) if mols else 0})
        motif_table = pd.DataFrame(motif_rows).sort_values(
            ['Molecules containing feature', 'Feature'], ascending=[False, True]
        )
        print('Motif occurrence among selected molecules (frequency is out of this small set):')
        display(motif_table)
        display(Draw.MolsToGridImage(
            mols, molsPerRow=2, subImgSize=(420, 300),
            legends=[f'Objective {score:.2f}' for score in top_docked['objective']]
        ))
